# Exploratory Wildfire Feature Analysis

This notebook extends the primary Missouri wildfire-susceptibility workflow with additional exploratory predictors, including fuel characteristics, soil properties, wind, drought, terrain, railroad proximity, and related geospatial variables.

**Important:** This notebook is exploratory and should not be treated as the primary model used for the final poster/report results.


In [ ]:
from collections import Counter
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyogrio
import seaborn as sns
import shap

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_selection import RFECV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier, XGBRegressor


In [ ]:
# Project paths
# Expected repository layout: notebooks/ and data/ at the repository root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
GDB1_PATH = DATA_DIR / "final_tables.gdb"
GDB2_PATH = DATA_DIR / "New_Wildfire_Finished.gdb"

for path in (GDB1_PATH, GDB2_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Place the required geodatabase in data/ "
            "or update the corresponding path variable."
        )


Load in both geodatabases and inspect their layers. Most preprocessing was done in ArcGIS Pro before loading the data into this notebook for further analysis. Categorical variables were the exception to this and required additional handling within the notebook.

In [ ]:
# Load both project geodatabases and inspect available layers.
gdb1 = {
    layer: gpd.read_file(GDB1_PATH, layer=layer, engine="pyogrio")
    for layer in pyogrio.list_layers(GDB1_PATH)[:, 0]
}

gdb2 = {
    layer: gpd.read_file(GDB2_PATH, layer=layer, engine="pyogrio")
    for layer in pyogrio.list_layers(GDB2_PATH)[:, 0]
}

print("GDB1 layers:")
for layer in gdb1:
    print(layer)

print("\nGDB2 layers:")
for layer in gdb2:
    print(layer)


NLCD is condensed into 8 broader categories and calculated as the proportion of each cell that is occupied by each category.
All work is done within the dictionary of geoDataFrames loaded from the geodatabases.

In [ ]:
# NLCD Feature Engineering

nlcd = gdb1["final_NLCD_landcover_areas_SqM"].copy()

nlcd.rename(columns={"CELL_ID_STABLE": "Cell_ID_Stable"}, inplace=True)

# consolidate NLCD classes into broader groups
nlcd_groups = {
    "Water": ["NLCD_water"],
    "Developed": [
        "NLCD_developed_open",
        "NLCD_developed_low",
        "NLCD_developed_medium",
        "NLCD_developed_high"
    ],
    "Barren": ["NLCD_barren"],
    "Forest": [
        "NLCD_deciduous_forest",
        "NLCD_evergreen_forest",
        "NLCD_mixed_forest"
    ],
    "Shrub": ["NLCD_shrub"],
    "Grassland": ["NLCD_grassland"],
    "Agriculture": [
        "NLCD_pasture",
        "NLCD_crop"
    ],
    "Wetlands": [
        "NLCD_woody_wetland",
        "NLCD_herbaceous_wetland"
    ]
}

# calculate total mapped land cover area
landcover_cols = [col for cols in nlcd_groups.values() for col in cols]
nlcd["NLCD_total_area"] = nlcd[landcover_cols].sum(axis=1)

# calculate proportion of each consolidated class
for group, cols in nlcd_groups.items():
    nlcd[f"{group}_Prop"] = (
        nlcd[cols].sum(axis=1) / nlcd["NLCD_total_area"]
    )

# keep only model-ready columns
gdb1["nlcd_prop"] = nlcd[
    ["Cell_ID_Stable"] +
    [f"{group}_Prop" for group in nlcd_groups]
].round(4)

del gdb1["final_NLCD_landcover_areas_SqM"]

FBFM40 (only 36 categories of which are present in Missouri) is condensed into 7 broader categories and calculated as the proportion of each cell that is occupied by each category.
All work is done within the dictionary of geoDataFrames loaded from the geodatabases.

In [ ]:
# FBFM40 Feature Engineering

# consolidate FBFM40 codes into broader groups
fbfm_groups = {
    "NB": [91, 93, 98, 99],
    "GR": [101, 102, 103, 104, 105, 106, 108],
    "GS": [121, 122, 123],
    "SH": [141, 142, 143, 144, 146, 147, 149],
    "TU": [161, 162, 163, 165],
    "TL": [181, 182, 183, 184, 185, 186, 188, 189],
    "SB": [201, 202, 203],
}

# calculate the proportion of each FBFM40 group relative to the cell area
for group, codes in fbfm_groups.items():
    cols = [f"VALUE_{code}" for code in codes]
    gdb2["fbfm40_Fishnet_Final"][f"{group}_prop"] = (
        gdb2["fbfm40_Fishnet_Final"][cols].sum(axis=1)
        / gdb2["fbfm40_Fishnet_Final"]["Cell_Area_M2"]
    )

# create a new DataFrame with only the relevant FBFM40 proportion columns
gdb2["fbfm40"] = gdb2["fbfm40_Fishnet_Final"][
    [
        "CELL_ID_STABLE",
        "NB_prop",
        "GR_prop",
        "GS_prop",
        "SH_prop",
        "TU_prop",
        "TL_prop",
        "SB_prop"
    ]
].copy()

# delete the original FBFM40 Fishnet DataFrame to free up memory
del gdb2["fbfm40_Fishnet_Final"]

Hydrologic Soil Group (HSG) is not consolidated because its categories represent unique hydrologic characteristics that are important for analysis. HSG is calculated as the proportion of each cell that is occupied by each category.
All work is done within the dictionary of geoDataFrames loaded from the geodatabases.

In [ ]:
# HSG Feature Engineering

# check which HSG classes are present in the data
print(gdb2["HSG_Fishnet_Final"]["HSG_CLEAN"].value_counts())

# list all columns in the HSG Fishnet DataFrame
gdb2["HSG_Fishnet_Final"].columns.tolist()

# check to make sure the percentages for each cell sum to 100
gdb2["HSG_Fishnet_Final"].groupby("Cell_ID_Stable")["PERCENTAGE"].sum().describe()

# pivot the HSG Fishnet DataFrame to have one row per cell and columns for each HSG class's percentage
hsg = gdb2["HSG_Fishnet_Final"].pivot_table(
    index="Cell_ID_Stable",
    columns="HSG_CLEAN",
    values="PERCENTAGE",
    fill_value=0
).reset_index()

# remove the hierarchical index created by the pivot operation
hsg.columns.name = None

# rename the HSG columns to indicate they are proportions
hsg.rename(columns={
    "A": "HSG_A_Prop",
    "A/D": "HSG_AD_Prop",
    "B": "HSG_B_Prop",
    "B/D": "HSG_BD_Prop",
    "C": "HSG_C_Prop",
    "C/D": "HSG_CD_Prop",
    "D": "HSG_D_Prop",
    "Unknown": "HSG_Unknown_Prop"
}, inplace=True)

# convert the HSG percentages to proportions
hsg_cols = [c for c in hsg.columns if c.startswith("HSG_")]
hsg[hsg_cols] = hsg[hsg_cols] / 100

# store the processed HSG DataFrame in the gdb2 dictionary and remove the original HSG Fishnet DataFrame
gdb2["hsg"] = hsg
del gdb2["HSG_Fishnet_Final"]

gdb2["hsg"][hsg_cols].sum(axis=1).describe()

Preprocessing: 

The two geodatabases must be cleaned prior to merging. Dataframes with identical or unintuitive column names are renamed for clarity, and Cell_ID_Stable is standardized across all DataFrames to ensure consistent merging.

Datasets unnecessary for the analysis, such as NLCD_majority_class, wui_class_areas_SqM_final, SVI_Cell_Summary_Final, and wildfires_final, are removed from the geodatabases prior to merging.

In [ ]:
# Check for duplicate column names across both geodatabases

all_columns = []

for gdb in [gdb1, gdb2]:
    for key, table in gdb.items():
        all_columns.extend(table.columns.tolist())

# find column names that occur more than once
duplicates = [
    col for col, count in Counter(all_columns).items()
    if count > 1
]

print(duplicates)

# show which dictionary/table contains each duplicate
for col in duplicates:
    print(f"\n{col}:")

    for gdb_name, gdb in [("gdb1", gdb1), ("gdb2", gdb2)]:
        for key, table in gdb.items():
            if col in table.columns:
                print(f"  {gdb_name}['{key}']")

# rename duplicate columns to avoid merging issues

gdb2["Wind_Mean_Fishnet_Final"].rename(
    columns={"MEAN": "Wind_Mean"},
    inplace=True
)

gdb1["elevation_mean_final"].rename(
    columns={"MEAN": "Elevation_Mean"},
    inplace=True
)

gdb1["prism_vpdmax_final"].rename(
    columns={"MEAN": "VPD_Max"},
    inplace=True
)

gdb1["prism_ppt_final"].rename(
    columns={"MEAN": "Precipitation_Mean"},
    inplace=True
)

gdb1["prism_tempmax_final"].rename(
    columns={"MEAN": "Temp_Max"},
    inplace=True
)

gdb1["prism_tempmean_final"].rename(
    columns={"MEAN": "Temp_Mean"},
    inplace=True
)

# remove old road density dataset
del gdb1["Road_Density_KM2"]

gdb2["RoadDensity_Final_Fishnet"].rename(
    columns={"Road_Density": "Road_Density"},
    inplace=True
)

# distinguish wildfire count periods
gdb2["Fires_32yrs_Fishnet_Final"].rename(
    columns={"Join_Count": "Wildfire_Count_32yr"},
    inplace=True
)

gdb2["Fires_10yrs_Fishnet_Final"].rename(
    columns={"Join_Count": "Wildfire_Count_10yr"},
    inplace=True
)

# make sure every table has the exact merge key name "Cell_ID_Stable"

for gdb_name, gdb in [("gdb1", gdb1), ("gdb2", gdb2)]:
    for key, table in gdb.items():
        if "Cell_ID_Stable" not in table.columns:
            print(f"{gdb_name}['{key}'] is missing Cell_ID_Stable")
            print(table.columns.tolist())

# fix column name for Cell_ID_Stable in fbfm40 table
gdb2["fbfm40"].rename(
    columns={"CELL_ID_STABLE": "Cell_ID_Stable"},
    inplace=True
)


# remove unnecessary columns and datasets before merging
for gdb_name, gdb in [("gdb1", gdb1), ("gdb2", gdb2)]:
    for key, table in gdb.items():
        print(f"\n{gdb_name}['{key}']")
        print(table.columns.tolist())

# Remove datasets that will not be used
del gdb1["NLCD_majority_class"]
del gdb1["wui_class_areas_SqM_final"]
del gdb1["SVI_Cell_Summary_Final"]
del gdb1["wildfires_final"]

gdb1 and gdb2 are merged based on the common key "Cell_ID_Stable" to create a unified dataset for further analysis. After this point, all operations are performed on "master".

In [ ]:
# The Great Merge

# define the fishnet dataframe as the master dataframe
master = gdb1["missouri_fishnet_final"].copy()

# merge all remaining tables from gdb1
for key, table in gdb1.items():
    if key != "missouri_fishnet_final":
        print(f"Merging gdb1['{key}']...")
        master = master.merge(
            table,
            on="Cell_ID_Stable",
            how="left"
        )

# merge all tables from gdb2
for key, table in gdb2.items():
    print(f"Merging gdb2['{key}']...")
    master = master.merge(
        table,
        on="Cell_ID_Stable",
        how="left"
    )

print(master.shape)
master.columns.tolist()

Most missing values were handled in ArcGIS Pro prior to feature engineering to ensure that all nulls were the result of known sliver cells or boundary issues, rather than genuine data gaps.

Two small cells are removed from the dataset due to small areas that prevent most predictors from contributing meaningful information to them. 

Fishnet centroids are calculated to perform nearest neighbor imputation on missing climate and AWS variables.

Aspect nulls were created in ArcGIS Pro to calculate mean aspect on non-flat terrain. Here, its null values are changed back to -1 to indicate flat terrain.

Population density missing values are assumed to be 0, as they correspond to areas with no population.

Housing density contained one null value, which was outside of the study area but was covered entirely by a wildlife conservation island, so its value is assumed to be 0.

Veg_2019, a vegetation cover variable, is considered redundant with the NLCD dataset and is therefore removed from the analysis.

In [ ]:
# check for missing values

# first remove the known sliver cells and cells with boundary issues that prevent meaningful data
master = master[master["Cell_ID_Stable"] != 3632].copy()
master = master[master["Cell_ID_Stable"] != 5702].copy()

# check remaining missing values
master.isna().sum().sort_values(ascending=False)


# NA handling

# load fishnet geometry from initial dataset load
fishnet_fc = gdb1["missouri_fishnet_featureclass"].copy()

# create centroid coordinates for nearest neighbor imputation
fishnet_centroids = fishnet_fc[["Cell_ID_Stable", "geometry"]].copy()

fishnet_centroids["x"] = fishnet_centroids.geometry.centroid.x
fishnet_centroids["y"] = fishnet_centroids.geometry.centroid.y

# merge centroid coordinates into master dataframe
master = master.merge(
    fishnet_centroids[["Cell_ID_Stable", "x", "y"]],
    on="Cell_ID_Stable",
    how="left"
)

# variables to be imputed using spatial nearest neighbor
nn_cols = [
    "Temp_Max",
    "Temp_Mean",
    "Precipitation_Mean",
    "VPD_Max",
    "AWS_MEAN_PERCELL",
    "Wind_Mean",
    "PDSI_DroughtMonths_Mean_Prop"
]

# separate complete and incomplete rows
missing = master[master[nn_cols].isna().any(axis=1)]
complete = master[master[nn_cols].notna().all(axis=1)]

# nearest neighbor model using fishnet centroid coordinates
nn = NearestNeighbors(n_neighbors=1)

nn.fit(complete[["x", "y"]])

_, nearest_idx = nn.kneighbors(
    missing[["x", "y"]]
)

# replace missing values with values from nearest complete cell
for col in nn_cols:
    master.loc[missing.index, col] = (
        complete.iloc[nearest_idx[:, 0]][col].values
    )

# no population density = 0
master["Population_Density"] = master["Population_Density"].fillna(0)

# flat terrain has undefined aspect
master["Aspect_Mean"] = master["Aspect_Mean"].fillna(-1)

# veg is redundant with NLCD anyways, remove column
master.drop(columns=["MEAN_VEG2019PC"], inplace=True)

# Housing density was a missing border cell but can be assumed 0
master["MEAN_HUDEN2020"] = master["MEAN_HUDEN2020"].fillna(0)

# drop centroid coordinates after imputation
master.drop(columns=["x", "y"], inplace=True)

# check for any remaining missing values
master.isna().sum().sort_values(ascending=False)


The binary target variable is created by determining which percentile cutoff point is most appropriate to identify cells that have a lower vs higher wildfire susceptibility based on 32 years of fire data. First, 5 different percentile cutoff points are evaluated to see how they separate low and elevated wildfire activity. Based on this analysis, a cutoff point of 6 historical ignitions is chosen as it represents the 75th percentile and provides a balanced distribution of observations across the two classes.

The new binary target is defined by assigning a value of 0 to cells with 0-5 historical ignitions (low wildfire activity) and a value of 1 to cells with 6 or more historical ignitions (elevated wildfire activity).

In [ ]:
# Exploratory binary target used in this extended-feature notebook.
# This threshold differs from the primary final model, which uses >=4 fires.
# 0 = fewer than 6 historical ignitions
# 1 = 6 or more historical ignitions
master["Risk_Binary"] = (
    master["Wildfire_Count_32yr"] >= 6
).astype(int)

print(master["Risk_Binary"].value_counts())
display(master.groupby("Risk_Binary")["Wildfire_Count_32yr"].describe())


A somewhat exploratory Random Forest model is constructed using all input features to evaluate their importance in predicting the binary wildfire susceptibility target variable. The model is trained using a Random Forest classifier, and feature selection is performed using Recursive Feature Elimination with Cross-Validation (RFECV) to identify the most relevant features for predicting wildfire susceptibility.

While RFECV indicated that 34 of 36 rows were important, the two missing rows belong to groups of features (FBFM and HSG) and are retained to calculate an overall feature importance for these groups.

In [ ]:
# create feature list
features = [
    "MEAN_HUDEN2020",
    "Elevation_Mean",
    "VPD_Max",
    "Precipitation_Mean",
    "Temp_Max",
    "Temp_Mean",
    "Population_Density",
    "Water_Prop",
    "Developed_Prop",
    "Barren_Prop",
    "Forest_Prop",
    "Shrub_Prop",
    "Grassland_Prop",
    "Agriculture_Prop",
    "Wetlands_Prop",
    "AWS_MEAN_PERCELL",
    "Aspect_Mean",
    "Slope_Mean",
    "Slope_STD",
    "Nearest_Rail",
    "Road_Density",
    "NB_prop",
    "GR_prop",
    "GS_prop",
    "SH_prop",
    "TU_prop",
    "TL_prop",
    "SB_prop",
    "HSG_A_Prop",
    "HSG_AD_Prop",
    "HSG_B_Prop",
    "HSG_BD_Prop",
    "HSG_C_Prop",
    "HSG_CD_Prop",
    "HSG_D_Prop",
    "HSG_Unknown_Prop",
    "Wind_Mean",
    "PDSI_DroughtMonths_Mean_Prop"
]

# Define X and y
X = master[features]
y = master["Risk_Binary"]

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

selector = RFECV(
    estimator=rf,
    step=1,
    cv=StratifiedKFold(5),
    scoring="f1",
    n_jobs=-1
)

selector.fit(X, y)

print("Optimal number of features:", selector.n_features_)

selected_features = X.columns[selector.support_]
print(selected_features)

An initial Random Forest classifier was constructed using all input features to evaluate their importance in predicting the binary wildfire susceptibility target. The model appears to be learning meaningful relationships between the predictors and the target, with precision and recall for high-susceptibility cells generally falling in the 60–70% range.

The model achieved an AUC of 0.867, indicating reasonably strong overall ability to distinguish between higher- and lower-susceptibility areas.

In [ ]:
X = master[features]
y = master["Risk_Binary"]


# train test split with the classic 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


rfb = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    max_features="sqrt",
    random_state=42,
    class_weight="balanced",
    bootstrap=True,
    n_jobs=-1
)

rfb.fit(X_train, y_train)

y_pred = rfb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))


# ROC AUC score
y_prob = rfb.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_prob)
print("\nAUC:", auc)

An initial feature importance analysis is conducted using all input features. AWS ranks highest, along with climate variables, FBFM, and human presence indicators like population density and road density.

In [ ]:
# RF feature importance binary

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rfb.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(20)

A correlation matrix is constructed to trim down excess features that are highly correlated with each other, helping to reduce multicollinearity and improve the interpretability of the model. 

Highly correlated features include the climate group, the human presence group, and NLCD with FBFM. 

In [ ]:
plt.figure(figsize=(20, 18))
sns.heatmap(
    master[features].corr(),
    cmap="coolwarm",
    center=0,
    annot=False,
    square=True,
    linewidths=0.5
)

plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

Correlated features are tested on RF models to evaluate their impact on model performance and to determine which features can be safely removed without significantly affecting predictive accuracy.

In [ ]:
# test removing nlcd or fbfm40 since they're generally highly correlated

# feature groups
nlcd_cols = [
    "Water_Prop",
    "Developed_Prop",
    "Barren_Prop",
    "Forest_Prop",
    "Shrub_Prop",
    "Grassland_Prop",
    "Agriculture_Prop",
    "Wetlands_Prop"
]
fbfm_cols = [
    "NB_prop",
    "GR_prop",
    "GS_prop",
    "SH_prop",
    "TU_prop",
    "TL_prop",
    "SB_prop"
]


# Model 1: exclude NLCD

features_no_nlcd = [
    col for col in features
    if col not in nlcd_cols
]

X_no_nlcd = master[features_no_nlcd]

X_train_nlcd, X_test_nlcd, y_train_nlcd, y_test_nlcd = train_test_split(
    X_no_nlcd,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf_no_nlcd = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    max_features="sqrt",
    random_state=42,
    class_weight="balanced",
    bootstrap=True,
    n_jobs=-1
)

rf_no_nlcd.fit(X_train_nlcd, y_train_nlcd)

y_pred_nlcd = rf_no_nlcd.predict(X_test_nlcd)
y_prob_nlcd = rf_no_nlcd.predict_proba(X_test_nlcd)[:, 1]

print("NO NLCD")
print("Accuracy:", accuracy_score(y_test_nlcd, y_pred_nlcd))
print(classification_report(y_test_nlcd, y_pred_nlcd))
print(confusion_matrix(y_test_nlcd, y_pred_nlcd))
print("AUC:", roc_auc_score(y_test_nlcd, y_prob_nlcd))



# Model 2: exclude FBFM40

features_no_fbfm = [
    col for col in features
    if col not in fbfm_cols
]

X_no_fbfm = master[features_no_fbfm]

X_train_fbfm, X_test_fbfm, y_train_fbfm, y_test_fbfm = train_test_split(
    X_no_fbfm,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf_no_fbfm = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    max_features="sqrt",
    random_state=42,
    class_weight="balanced",
    bootstrap=True,
    n_jobs=-1
)

rf_no_fbfm.fit(X_train_fbfm, y_train_fbfm)

y_pred_fbfm = rf_no_fbfm.predict(X_test_fbfm)
y_prob_fbfm = rf_no_fbfm.predict_proba(X_test_fbfm)[:, 1]

print("\nNO FBFM40")
print("Accuracy:", accuracy_score(y_test_fbfm, y_pred_fbfm))
print(classification_report(y_test_fbfm, y_pred_fbfm))
print(confusion_matrix(y_test_fbfm, y_pred_fbfm))
print("AUC:", roc_auc_score(y_test_fbfm, y_prob_fbfm))

# NLCD can be dropped from model

In [ ]:
# test different combinations of huden, popden, roadden

# human-development/access variables to compare
human_vars = [
    "Population_Density",
    "MEAN_HUDEN2020",
    "Road_Density"
]

# all other predictors stay fixed
base_features = [
    col for col in features
    if col not in human_vars
]

# feature combinations to test
human_tests = {
    "All_Three": [
        "Population_Density",
        "MEAN_HUDEN2020",
        "Road_Density"
    ],
    "Population_Only": [
        "Population_Density"
    ],
    "Housing_Only": [
        "MEAN_HUDEN2020"
    ],
    "Road_Only": [
        "Road_Density"
    ],
    "Population_Road": [
        "Population_Density",
        "Road_Density"
    ],
    "Population_Housing": [
        "Population_Density",
        "MEAN_HUDEN2020"
    ],
    "Housing_Road": [
        "MEAN_HUDEN2020",
        "Road_Density"
    ]
}

results = []

for name, human_features in human_tests.items():

    test_features = base_features + human_features

    X = master[test_features]
    y = master["Risk_Binary"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    rf_test = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        max_features="sqrt",
        random_state=42,
        class_weight="balanced",
        bootstrap=True,
        n_jobs=-1
    )

    rf_test.fit(X_train, y_train)

    y_pred = rf_test.predict(X_test)
    y_prob = rf_test.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob)
    })

human_results = pd.DataFrame(results).sort_values(
    "AUC",
    ascending=False
)

human_results

# population density and housing density can both probably be scrapped

In [ ]:
# compare highly correlated terrain variables

# terrain variables to compare
terrain_vars = [
    "Elevation_Mean",
    "Aspect_Mean",
    "Slope_Mean",
    "Slope_STD"
]

# all other predictors stay fixed
base_features = [
    col for col in features
    if col not in terrain_vars
]

# terrain combinations to test
terrain_tests = {
    "All_Terrain": [
        "Elevation_Mean",
        "Aspect_Mean",
        "Slope_Mean",
        "Slope_STD"
    ],
    "Elevation_Only": [
        "Elevation_Mean"
    ],
    "Aspect_Only": [
        "Aspect_Mean"
    ],
    "Slope_Only": [
        "Slope_Mean",
        "Slope_STD"
    ],
    "Elevation_Slope": [
        "Elevation_Mean",
        "Slope_Mean",
        "Slope_STD"
    ],
    "Elevation_Aspect": [
        "Elevation_Mean",
        "Aspect_Mean"
    ],
    "Slope_Aspect": [
        "Slope_Mean",
        "Slope_STD",
        "Aspect_Mean"
    ]
}

results = []

for name, terrain_features in terrain_tests.items():

    test_features = base_features + terrain_features

    X = master[test_features]
    y = master["Risk_Binary"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    rf_test = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        max_features="sqrt",
        random_state=42,
        class_weight="balanced",
        bootstrap=True,
        n_jobs=-1
    )

    rf_test.fit(X_train, y_train)

    y_pred = rf_test.predict(X_test)
    y_prob = rf_test.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob)
    })

terrain_results = pd.DataFrame(results).sort_values(
    "AUC",
    ascending=False
)

terrain_results

# We can remove all terrain variables except elevation mean

In [ ]:
climate_vars = [
    "VPD_Max",
    "Precipitation_Mean",
    "Temp_Max",
    "Temp_Mean"
]

# all non-climate features stay fixed
base_features = [
    col for col in new_features
    if col not in climate_vars
]

shap_results = []

# full climate model + leave-one-out models
climate_tests = {
    "All_Climate": climate_vars
}

for var in climate_vars:
    climate_tests[f"No_{var}"] = [
        col for col in climate_vars
        if col != var
    ]

for name, climate_features in climate_tests.items():

    test_features = base_features + climate_features

    X = master[test_features]
    y = master["Risk_Binary"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    rf_test = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        max_features="sqrt",
        random_state=42,
        class_weight="balanced",
        bootstrap=True,
        n_jobs=-1
    )

    rf_test.fit(X_train, y_train)

    # SHAP
    explainer = shap.TreeExplainer(rf_test)
    shap_values = explainer.shap_values(X_test)

    # class 1 SHAP values
    if isinstance(shap_values, list):
        class1_shap = shap_values[1]
    else:
        class1_shap = shap_values[:, :, 1]

    mean_abs_shap = abs(class1_shap).mean(axis=0)

    for feature, importance in zip(test_features, mean_abs_shap):
        if feature in climate_vars:
            shap_results.append({
                "Model": name,
                "Climate_Feature": feature,
                "Mean_Abs_SHAP": importance
            })

climate_shap_results = pd.DataFrame(shap_results)

climate_shap_results

# we can safely remove VPD_Max and Temp_Mean

A new random forest model is tested on a set of features that removes NLCD, population density, housing density, slope mean, slope max, aspect mean, max VPD, and mean temperature.

In [ ]:
# test an rf model with an updated feature set

new_features = [
    "Elevation_Mean",
    "Precipitation_Mean",
    "Temp_Max",
    "AWS_MEAN_PERCELL",
    "Nearest_Rail",
    "Road_Density",
    "NB_prop",
    "GR_prop",
    "GS_prop",
    "SH_prop",
    "TU_prop",
    "TL_prop",
    "SB_prop",
    "HSG_A_Prop",
    "HSG_AD_Prop",
    "HSG_B_Prop",
    "HSG_BD_Prop",
    "HSG_C_Prop",
    "HSG_CD_Prop",
    "HSG_D_Prop",
    "HSG_Unknown_Prop",
    "Wind_Mean",
    "PDSI_DroughtMonths_Mean_Prop"
]

X = master[new_features]
y = master["Risk_Binary"]


# train test split with the classic 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


rfb = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    max_features="sqrt",
    random_state=42,
    class_weight="balanced",
    bootstrap=True,
    n_jobs=-1
)

rfb.fit(X_train, y_train)

y_pred = rfb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))


# ROC AUC score
y_prob = rfb.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_prob)
print("\nAUC:", auc)

In [ ]:
plt.figure(figsize=(20, 18))
sns.heatmap(
    master[new_features].corr(),
    cmap="coolwarm",
    center=0,
    annot=False,
    square=True,
    linewidths=0.5
)

plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# generate predictions for all cells
master["Predicted_Probability"] = rfb.predict_proba(
    master[new_features]
)[:, 1]

master["Predicted_Class"] = rfb.predict(
    master[new_features]
)

map_gdf = gdb1["missouri_fishnet_featureclass"].merge(
    master[
        [
            "Cell_ID_Stable",
            "Wildfire_Count_32yr",
            "Risk_Binary",
            "Predicted_Probability",
            "Predicted_Class"
        ]
    ],
    on="Cell_ID_Stable",
    how="inner"
)

ax = map_gdf.plot(
    column="Predicted_Probability",
    figsize=(10, 8),
    legend=True,
    edgecolor="none",
    cmap="RdBu_r",
    vmin=0,
    vmax=1
)

ax.set_title("Predicted Wildfire Susceptibility")
ax.axis("off")

plt.show()

In [ ]:
ax = map_gdf.plot(
    column="Wildfire_Count_32yr",
    figsize=(10, 8),
    legend=True,
    edgecolor="none",
    cmap="RdBu_r",
    scheme="quantiles",
    k=5
)

ax.set_title("Observed 32-Year Wildfire Counts")
ax.axis("off")

plt.show()

In [ ]:
new_features_df = master[new_features]

new_features_df.columns.tolist()